# Assignment 2, Part 2: Pedestrians on Eindhoven Centraal

In this notebook you model the future motion of pedestrians on platforms 3 and 4 of Eindhoven Centraal. Given the first 10 observed frames of a scene, you generate the remaining 20 frames for all pedestrians using conditional flow matching with time bundling.

You implement and compare two models:
1. a model with an uninformed Gaussian prior on the future velocities;
2. a model with an informed, data dependent prior.

The focus is not on model performance, but on an architecture that fits the problem formulation and on the comparison between the two priors. Every group member must be able to explain the model design and interpret the results. The full task description is in Section 2 of the assignment PDF.

In [ ]:
import gdown
from pathlib import Path
import pickle
import torch
from torch.utils.data import Dataset
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from lightning import LightningDataModule, LightningModule, Trainer
import math
import warnings
import matplotlib.pyplot as plt
import numpy as np

## Dataset

The dataset contains 100,000 samples. Each sample is a scene with 5 to 100 pedestrians, stored as an array of shape `(N, 30, 2)`: the 2D positions in metres of `N` pedestrians over 30 frames, spaced $\Delta t = 0.5$ s apart (15 s in total). Every pedestrian is present in all 30 frames.

The cell below downloads the data (about 290 MB).

In [ ]:
drive_link = "https://drive.google.com/file/d/1sU_gti1xg2AB_SQINy_cZZm4jxE9lbIm/view?usp=drive_link"

DATA_PATH = Path("eindhoven_centraal.pkl")

if not DATA_PATH.exists():

    print("Downloading dataset...")

    gdown.download(
        url=drive_link,
        output=str(DATA_PATH),
        quiet=False,
    )

    print(
        "Download complete:",
        DATA_PATH,
    )

else:
    print(
        "Dataset already available:",
        DATA_PATH,
    )

## Data loading and graph construction

`PedestriansDataset` loads the samples and turns each one into a graph:

* **Nodes** are pedestrians. The node features `x` are the full trajectories, with shape `(N, 30, 2)`.
* **Edges** are fixed over time: the same `edge_index` of shape `(2, E)` is used for every frame. The example graph is fully connected without self loops, so $E = N(N-1)$.

After batching with the PyG `DataLoader`, `x` has shape `(sum N, 30, 2)`, the edges of each scene are offset so they never connect different scenes, and `batch` gives the scene index of every pedestrian.

A fully connected graph grows with $N^2$ and connects pedestrians that are far apart and do not interact. You can and should implement a less connected graph in `make_edge_index`, for example a radius or k nearest neighbour graph. Such a graph may only be built from the observed frames 0 to 9, since the target frames are not available at inference.

`GraphDataModule` splits the samples randomly at sample level with a fixed seed, so no scene appears in more than one split. The test set must contain at least 1,000 samples; a larger test set is preferred. Normalization statistics must be computed on the training split only.

In [ ]:
class PedestriansDataset(Dataset):
    """Pedestrian scenes as graphs with an edge connectivity that is fixed over time.

    Every sample is one scene with N pedestrians over F = 30 frames. Nodes are pedestrians
    and the node features are their full trajectories:
        x:          (N, F, 2)   positions in metres
        edge_index: (2, E)      edges between pedestrians, shared by all frames

    After batching with the PyG DataLoader:
        x:          (sum N, F, 2)
        edge_index: (2, sum E)  offset per scene, so edges never cross scenes
        batch:      (sum N,)    scene index of every pedestrian
    """

    def __init__(self, path, n_samples=None, T_cond=10):
        self.T_cond = T_cond
        self.raw_data = self._load_data(path)
        if n_samples is not None:
            self.raw_data = self.raw_data[:n_samples]
        print(f"Using {len(self.raw_data)} samples.")

    def _load_data(self, path):
        with open(path, "rb") as f:
            data = pickle.load(f)
        data = [torch.tensor(sample, dtype=torch.float32) for sample in data]
        print(
            f"\nLoaded {len(data)} samples from {path}, containing in total "
            f"{sum(x.shape[0] for x in data)} trajectories of {data[0].shape[1]} frames.\n"
        )
        return data

    def make_edge_index(self, sample):
        """Fully connected graph between N pedestrians, without self loops. Shape: (2, N * (N - 1))."""
        # NOTE: A fully connected graph grows with N^2 and also connects pedestrians that are far
        # apart and do not interact. A less connected graph can and should be implemented here,
        # for example a radius graph or a k nearest neighbour graph. To avoid leaking information
        # about the future, such a graph may only be built from the observed frames 0 to T_cond - 1.
        N = sample.shape[0]
        not_self = ~torch.eye(N, dtype=torch.bool)  # True everywhere except the diagonal
        return not_self.nonzero().T.contiguous()

    def __len__(self):
        return len(self.raw_data)

    def __getitem__(self, idx):
        sample = self.raw_data[idx]  # (N, F, 2)
        N = sample.shape[0]
        return Data(x=sample, edge_index=self.make_edge_index(sample), num_nodes=N)


class GraphDataModule(LightningDataModule):
    """Random split at sample level, so no scene appears in more than one split."""

    def __init__(self, dataset, train_split, val_split, test_split, batch_size=8, seed=42, min_test_size=1000):
        super().__init__()
        if not math.isclose(train_split + val_split + test_split, 1.0):
            raise ValueError("train_split, val_split and test_split must sum to 1.")
        self.batch_size = batch_size
        self.seed = seed
        self.train_data, self.val_data, self.test_data = self.split_data(dataset, train_split, val_split)

        print(
            f"Split with seed {seed}: {len(self.train_data)} train, "
            f"{len(self.val_data)} validation, {len(self.test_data)} test samples."
        )
        if len(self.test_data) < min_test_size:
            warnings.warn(
                f"The test set contains {len(self.test_data)} samples, "
                f"but the assignment requires at least {min_test_size}."
            )

    def split_data(self, dataset, train_split, val_split):
        total_size = len(dataset)
        train_size = int(total_size * train_split)
        val_size = int(total_size * val_split)
        test_size = total_size - train_size - val_size

        return torch.utils.data.random_split(
            dataset,
            [train_size, val_size, test_size],
            generator=torch.Generator().manual_seed(self.seed),
        )

    def train_dataloader(self):
        return DataLoader(self.train_data, batch_size=self.batch_size, shuffle=True)  # type: ignore

    def val_dataloader(self):
        return DataLoader(self.val_data, batch_size=self.batch_size)  # type: ignore

    def test_dataloader(self):
        return DataLoader(self.test_data, batch_size=self.batch_size)  # type: ignore

    def predict_dataloader(self):
        return DataLoader(self.test_data, batch_size=self.batch_size)  # type: ignore

## Configuration

`n_samples` limits the number of samples for faster development. Make sure the final test set contains at least 1,000 samples, and report the seed and the size of each split.

In [ ]:
T_cond = 10

n_samples = 10000  # Limit to a number of samples for faster training during development

dataset = PedestriansDataset(
    DATA_PATH, n_samples=n_samples, T_cond=T_cond
) 

data_module = GraphDataModule(
    dataset, train_split=0.7, val_split=0.1, test_split=0.2, batch_size=16
)

## Visualization
This method can be used to visualize the pedestrian trajectories. Use it to double check if your generated samples are realistic.

Make sure you download the background image: https://github.com/vlamen/ml4sci-2026/blob/main/assignments/Eindhoven_centraal_platform_3_4.png

In [ ]:
EXTENT = (-6.309, 75.091, -16.539, 10.909)  # (left, right, bottom, top) of the background image in metres


def plot_sample(sample, background_path="Eindhoven_centraal_platform_3_4.png", ax=None,
                cmap_name="tab20", figsize=(12, 4.5), dpi=150):
    """Plot all trajectories of one data sample on the station background.

    sample: (N, F, 2) positions of N pedestrians over F frames, as a numpy array or torch tensor.
    The x and y axes of the data are swapped to match the background image.
    Each pedestrian gets its own color; the dot marks the final position.
    """
    if isinstance(sample, torch.Tensor):
        sample = sample.detach().cpu().numpy()
    traj = np.asarray(sample)[..., [1, 0]]  # Swap x and y

    if ax is None:
        fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
    else:
        fig = ax.figure

    ax.imshow(plt.imread(background_path), extent=EXTENT, origin="upper", zorder=0)
    ax.set_xlim(EXTENT[0], EXTENT[1])
    ax.set_ylim(EXTENT[2], EXTENT[3])

    cmap = plt.get_cmap(cmap_name)
    for i, t in enumerate(traj):
        color = cmap(i % cmap.N)
        ax.plot(t[:, 0], t[:, 1], color=color, lw=1.5, zorder=1)
        ax.scatter(t[-1, 0], t[-1, 1], color=color, s=15, edgecolors="black", linewidths=0.4, zorder=2)

    ax.set_title(f"{traj.shape[0]} pedestrians, {traj.shape[1]} frames")
    return fig, ax

plot_sample(dataset[0].x)
plt.show()

## Model architecture

Your model must capture patterns over **space**, the interactions between pedestrians, and over **time**, how the motion of each pedestrian evolves over the frames. As discussed in Practical 4, message passing handles space but not time, and flattening all frames into an MLP ignores temporal order and locality. Design an architecture that handles both axes, for example message passing over pedestrians combined with convolutions or attention over time.

Use the same architecture, training budget and sampler for both models, so that the prior is the only difference between them.

## Part I: Uninformed Gaussian prior

Train a conditional flow matching model that generates the velocities of frames 10 to 29, conditioned on frames 0 to 9, using the independent coupling

$$\pi(x_0, x_1 \mid c) = p_0(x_0)\,p_1(x_1 \mid c), \qquad p_0 = \mathcal{N}(0, \sigma^2 I).$$

All 20 frames are generated in a single model call. Positions are recovered by summing the generated velocities, starting from the last observed position.

## Part II: Informed prior

Train a second model using a data dependent coupling

$$\pi(x_0, x_1 \mid c) = p_0(x_0 \mid c)\,p_1(x_1 \mid c).$$

The prior may only use the observed frames 0 to 9, so that $x_0 \perp x_1 \mid c$. It must remain stochastic and be sampled in the same way during training and inference.

## Evaluation

For every test sample, generate $M = 20$ futures with each model. All metrics are computed on frames 10 to 29 only.

**Pointwise metrics.** Report ADE, FDE, $\text{min}_{20}$ ADE and $\text{min}_{20}$ FDE. ADE and FDE are averaged over all pedestrians, samples and generated futures. For the $\text{min}_{20}$ metrics, the best future is selected per data sample based on the error averaged over all its pedestrians, not per pedestrian.

**Distributional metrics.** Plot histogram densities of the ground truth and both models for velocity magnitude, acceleration magnitude, tortuosity and distance to the closest neighbour within the same frame.

The exact definitions are given in the evaluation section of the assignment PDF. You are of course allowed to use code from the practicals

## Comparison

Show the four pointwise metrics for both models in a table, together with the four histograms. Explanations and interpretations of the results are written in Ans, not in this notebook.